# Multi-Source Data Fusion for Air Quality Index Forecasting Using Time-Series Models
**Companion notebook for the manuscript.** Runs end-to-end in Google Colab (CPU is enough; GPU only for the optional GRU cell).

**Pipeline:** CPCB `city_day.csv` -> cleaning -> leakage-free features -> baselines and ML models -> walk-forward + extreme-day analysis -> feature-group ablation -> grouped SHAP -> *(fusion extension)* ERA5 weather + AOD + fire features.

**Files you need in Google Drive (`MyDrive`)**
1. `city_day.csv` (Kaggle: *Air Quality Data in India 2015-2020*) - required.
2. `era5_daily.csv` - ERA5 weather, one row per city and day (`City, Date, t2m, rh, ws, u10, v10, sp, tp`). Section 9a creates it for you from the free Open-Meteo archive API (no key needed).
3. `aod_daily.csv` - optional, columns: `City, Date, aod`.
4. `fire_daily.csv` - optional, columns: `City, Date, fire_count`.

**Leakage rules used throughout:** (i) time-ordered split (train 2015-2018, validation 2019, test 2020); (ii) every feature uses information up to day *t* only, target is AQI(*t*+h); (iii) imputation statistics are fitted on the training rows of each model only; (iv) AQI values above 500 (outside the official scale) are set to missing; (v) weather/AOD/fire features are lagged to day *t* (no future observations).

In [ ]:
# ---- 0. Setup ----
!pip -q install xgboost lightgbm shap statsmodels
import os
try:
    from google.colab import drive
    drive.mount('/content/drive'); BASE = "/content/drive/MyDrive"
except Exception:
    BASE = "."          # running locally

In [ ]:
# ---- 1. Configuration ----
QUICK = False      # True = smoke test: first city only, fewer seeds (about 3 minutes)
CITY_DAY_PATH = f"{BASE}/city_day.csv"
WEATHER_PATH  = f"{BASE}/era5_daily.csv"
AOD_PATH      = f"{BASE}/aod_daily.csv"
FIRE_PATH     = f"{BASE}/fire_daily.csv"
OUT = f"{BASE}/aqi_fusion_results"; os.makedirs(OUT, exist_ok=True)

## 2. Core library (feature builder, models, metrics, Diebold-Mariano test)

In [ ]:
"""Core library for the AQI forecasting experiments (pollutant-only feature set A, extendable with weather/AOD)."""
import numpy as np, pandas as pd, warnings
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb, lightgbm as lgb
warnings.filterwarnings("ignore")

CITIES = ["Delhi", "Lucknow", "Patna", "Chennai", "Hyderabad"]
POLL_ALL = ["PM2.5","PM10","NO","NO2","NOx","NH3","CO","SO2","O3","Benzene","Toluene","Xylene"]
LAGS = [0, 1, 2, 6]
T_TRAIN_END, T_VAL_END = pd.Timestamp("2018-12-31"), pd.Timestamp("2019-12-31")
EXTREME = 300
DEF_XGB = dict(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.85,
               colsample_bytree=0.8, min_child_weight=3, reg_lambda=5)

def load_city(df, city):
    d = df[df.City == city].drop_duplicates("Date").set_index("Date").sort_index()
    d = d.reindex(pd.date_range(d.index.min(), d.index.max(), freq="D"))
    cols = ["AQI"] + POLL_ALL
    d = d[cols].astype(float)
    n_inv = int((d["AQI"] > 500).sum())          # AQI scale tops out at 500 -> invalid
    d = d.mask(d < 0)
    d.loc[d["AQI"] > 500, "AQI"] = np.nan
    return d, n_inv

def select_pollutants(d, thr=0.40):
    tr = d[:T_TRAIN_END]
    return [p for p in POLL_ALL if tr[p].isna().mean() < thr]

def build(d, pollutants, h, extra=None):
    """Leakage-free features: only information up to day t; target = AQI(t+h)."""
    vars_ = ["AQI"] + pollutants
    f = d[vars_].ffill(limit=3)
    feats = {}
    for v in vars_:
        for l in LAGS: feats[f"{v}_lag{l}"] = f[v].shift(l)
        feats[f"{v}_r3"] = f[v].rolling(3, min_periods=1).mean()
        feats[f"{v}_r7"] = f[v].rolling(7, min_periods=1).mean()
        feats[f"{v}_miss"] = d[v].isna().astype(float)
    X = pd.DataFrame(feats, index=d.index)
    if extra is not None:                       # weather / AOD columns, already lagged to day t
        X = X.join(extra, how="left")
    tdate = X.index + pd.Timedelta(days=h)
    X["month_sin"] = np.sin(2*np.pi*tdate.month/12); X["month_cos"] = np.cos(2*np.pi*tdate.month/12)
    X["dow_sin"] = np.sin(2*np.pi*tdate.dayofweek/7); X["dow_cos"] = np.cos(2*np.pi*tdate.dayofweek/7)
    X["y"] = d["AQI"].shift(-h)
    X["tdate"] = tdate
    X = X[X["y"].notna() & X["AQI_lag0"].notna()]
    return X

def group_of(col, pollutants, extra_prefixes=()):
    if col.endswith("_miss"): return "Missing flags"
    if col.startswith(("month", "dow")): return "Calendar"
    if col.startswith("AQI_"): return "AQI history"
    for name, pref in extra_prefixes:
        if col.startswith(pref): return name
    for p in pollutants:
        if col.startswith(p + "_"): return p
    return "Other"

def split_masks(X):
    t = X["tdate"]
    return (t <= T_TRAIN_END), ((t > T_TRAIN_END) & (t <= T_VAL_END)), (t > T_VAL_END)

def feat_cols(X): return [c for c in X.columns if c not in ("y", "tdate")]

def impute(Xtr, *others):
    med = Xtr.median().fillna(0)
    return [Xtr.fillna(med)] + [o.fillna(med) for o in others]

def reg(y, p):
    return dict(RMSE=float(np.sqrt(mean_squared_error(y, p))), MAE=float(mean_absolute_error(y, p)), R2=float(r2_score(y, p)))

def ext(y, p, thr=EXTREME):
    yt, yp = np.asarray(y) > thr, np.asarray(p) > thr
    tp, fp, fn = int((yt & yp).sum()), int((~yt & yp).sum()), int((yt & ~yp).sum())
    prec = tp/(tp+fp) if tp+fp else np.nan
    rec = tp/(tp+fn) if tp+fn else np.nan
    f1 = 2*prec*rec/(prec+rec) if (prec==prec and rec==rec and prec+rec>0) else np.nan
    return dict(support=int(yt.sum()), TP=tp, FP=fp, FN=fn, precision=prec, recall=rec, F1=f1)

def fit_xgb(cfg, X, y, seed=0, w=None):
    return xgb.XGBRegressor(**cfg, random_state=seed, n_jobs=1, tree_method="hist").fit(X, y, sample_weight=w)

def tune_xgb(Xtr, ytr, Xva, yva, n_iter=15, seed=0):
    rng = np.random.RandomState(seed)
    best, best_cfg = 1e18, DEF_XGB
    for _ in range(n_iter):
        cfg = dict(n_estimators=int(rng.choice([200, 400, 600])), max_depth=int(rng.choice([3, 4, 5, 6])),
                   learning_rate=float(rng.choice([0.02, 0.05, 0.1])), subsample=float(rng.choice([0.7, 0.85, 1.0])),
                   colsample_bytree=float(rng.choice([0.6, 0.8, 1.0])), min_child_weight=int(rng.choice([1, 3, 5])),
                   reg_lambda=float(rng.choice([1, 5, 10])))
        m = fit_xgb(cfg, Xtr, ytr, seed=0)
        r = np.sqrt(mean_squared_error(yva, m.predict(Xva)))
        if r < best: best, best_cfg = r, cfg
    return best_cfg, best

def ridge_fit_predict(Xtr, ytr, Xte_list, alpha):
    sc = StandardScaler().fit(Xtr)
    m = Ridge(alpha=alpha).fit(sc.transform(Xtr), ytr)
    return [m.predict(sc.transform(x)) for x in Xte_list]

def mlp_fit_predict(Xtr, ytr, Xte_list, seed):
    sc = StandardScaler().fit(Xtr); ym, ys = ytr.mean(), ytr.std()
    m = MLPRegressor(hidden_layer_sizes=(64, 32), alpha=1e-3, early_stopping=True, validation_fraction=0.15,
                     max_iter=500, random_state=seed).fit(sc.transform(Xtr), (ytr-ym)/ys)
    return [m.predict(sc.transform(x))*ys+ym for x in Xte_list]

def rf_model(seed): return RandomForestRegressor(n_estimators=300, min_samples_leaf=3, max_features=0.5, n_jobs=1, random_state=seed)
def lgb_model(seed): return lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=15, subsample=0.85, subsample_freq=1,
                                              colsample_bytree=0.8, min_child_samples=10, reg_lambda=5, random_state=seed, n_jobs=1, verbose=-1)

def dm_test(e_base, e_model, h=1):
    """Diebold-Mariano (squared-error loss) with Harvey-Leybourne-Newbold correction. Positive stat => model better than base."""
    d = np.asarray(e_base)**2 - np.asarray(e_model)**2
    n = len(d); dbar = d.mean(); s = np.var(d, ddof=0)
    for k in range(1, h):
        s += 2*np.cov(d[k:], d[:-k], ddof=0)[0, 1]
    stat = dbar/np.sqrt(s/n) * np.sqrt((n+1-2*h+h*(h-1)/n)/n)
    return float(stat), float(2*(1-stats.t.cdf(abs(stat), df=n-1)))

## 3. Load data, choose cities, dataset summary

In [ ]:
df = pd.read_csv(CITY_DAY_PATH, parse_dates=["Date"])
CITY_LIST = CITIES[:1] if QUICK else CITIES        # Delhi, Lucknow, Patna, Chennai, Hyderabad (Ahmedabad excluded: AQI > 500 inconsistencies)
NSEED = 2 if QUICK else 5
R = {}                                             # all result tables are collected here
def add(name, row): R.setdefault(name, []).append(row)

def prepare(city, h=1, extra=None):
    d, n_inv = load_city(df, city); pol = select_pollutants(d)
    X = build(d, pol, h, extra=extra); cols = feat_cols(X)
    mtr, mva, mte = split_masks(X)
    P = dict(city=city, d=d, pol=pol, n_inv=n_inv, X=X, cols=cols, mtr=mtr, mva=mva, mte=mte,
             Xtr=X.loc[mtr, cols], Xva=X.loc[mva, cols], Xte=X.loc[mte, cols],
             ytr=X.loc[mtr, "y"].values, yva=X.loc[mva, "y"].values, yte=X.loc[mte, "y"].values)
    P["Xtr_i"], P["Xva_i"], P["Xte_i"] = impute(P["Xtr"], P["Xva"], P["Xte"])          # fitted on train only
    P["Xtv_i"], P["Xte_i2"] = impute(pd.concat([P["Xtr"], P["Xva"]]), P["Xte"])         # refit set (train+val)
    P["ytv"] = np.concatenate([P["ytr"], P["yva"]])
    return P

PREP = {c: prepare(c) for c in CITY_LIST}
for c, P in PREP.items():
    raw = P["d"]["AQI"]
    add("summary", dict(city=c, start=str(P["d"].index.min().date()), end=str(P["d"].index.max().date()), days=len(P["d"]),
        aqi_missing_pct=round(raw.isna().mean()*100, 1), invalid_gt500=P["n_inv"],
        ext_train=int((raw[:T_TRAIN_END] > EXTREME).sum()), ext_val=int((raw["2019-01-01":"2019-12-31"] > EXTREME).sum()),
        ext_test=int((raw["2020-01-01":] > EXTREME).sum()), pollutants=",".join(P["pol"])))
pd.DataFrame(R["summary"])

## 4. Models on the fixed split (h = 1 day): baselines, Ridge, RF, LightGBM, MLP, XGBoost (tuned)

In [ ]:
FIXED_PRED = {}
for city, P in PREP.items():
    yva, yte = P["yva"], P["yte"]; pv, pt, sd = {}, {}, {}
    pv["Persistence"], pt["Persistence"] = P["Xva"]["AQI_lag0"].values, P["Xte"]["AQI_lag0"].values
    pv["Seasonal naive (7 d)"] = P["Xva"]["AQI_lag6"].fillna(P["Xva"]["AQI_lag0"]).values
    pt["Seasonal naive (7 d)"] = P["Xte"]["AQI_lag6"].fillna(P["Xte"]["AQI_lag0"]).values
    best_a, best_r = 1, 1e18
    for a in [0.1, 1, 10, 100, 1000]:
        r = np.sqrt(mean_squared_error(yva, ridge_fit_predict(P["Xtr_i"], P["ytr"], [P["Xva_i"]], a)[0]))
        if r < best_r: best_a, best_r = a, r
    pv["Ridge (AR)"] = ridge_fit_predict(P["Xtr_i"], P["ytr"], [P["Xva_i"]], best_a)[0]
    pt["Ridge (AR)"] = ridge_fit_predict(P["Xtv_i"], P["ytv"], [P["Xte_i2"]], best_a)[0]
    for name, mk, ns in [("Random Forest", rf_model, 3), ("LightGBM", lgb_model, NSEED)]:
        pv[name] = np.mean([mk(s).fit(P["Xtr_i"], P["ytr"]).predict(P["Xva_i"]) for s in range(2)], axis=0)
        L = [mk(s).fit(P["Xtv_i"], P["ytv"]).predict(P["Xte_i2"]) for s in range(ns)]
        pt[name] = np.mean(L, axis=0); sd[name] = float(np.std([np.sqrt(mean_squared_error(yte, p)) for p in L]))
    pv["MLP"] = np.mean([mlp_fit_predict(P["Xtr_i"], P["ytr"], [P["Xva_i"]], s)[0] for s in range(2)], axis=0)
    L = [mlp_fit_predict(P["Xtv_i"], P["ytv"], [P["Xte_i2"]], s)[0] for s in range(3)]
    pt["MLP"] = np.mean(L, axis=0); sd["MLP"] = float(np.std([np.sqrt(mean_squared_error(yte, p)) for p in L]))
    cfg, vr = tune_xgb(P["Xtr_i"], P["ytr"], P["Xva_i"], yva, n_iter=6 if QUICK else 15)
    pv["XGBoost"] = fit_xgb(cfg, P["Xtr_i"], P["ytr"], 0).predict(P["Xva_i"])
    L = [fit_xgb(cfg, P["Xtv_i"], P["ytv"], s).predict(P["Xte_i2"]) for s in range(NSEED)]
    pt["XGBoost"] = np.mean(L, axis=0); sd["XGBoost"] = float(np.std([np.sqrt(mean_squared_error(yte, p)) for p in L]))
    add("tuned", dict(city=city, **cfg, val_rmse=vr, ridge_alpha=best_a)); FIXED_PRED[city] = pt
    for m in pt:
        add("fixed", dict(city=city, model=m, n_val=len(yva), n_test=len(yte), **{f"val_{k}": v for k, v in reg(yva, pv[m]).items()},
                          **{f"test_{k}": v for k, v in reg(yte, pt[m]).items()}, test_RMSE_std=sd.get(m, 0.0)))
    for base in ["Persistence", "Ridge (AR)"]:
        st, p = dm_test(yte - pt[base], yte - pt["XGBoost"], 1); add("dm", dict(city=city, base=base, stat=st, p=p, n=len(yte)))
    td = pd.to_datetime(P["X"].loc[P["mte"], "tdate"].values)
    for pn, msk in {"Pre-lockdown (1 Jan-24 Mar)": td < "2020-03-25", "Lockdown (25 Mar-31 May)": (td >= "2020-03-25") & (td <= "2020-05-31"), "Unlock (1 Jun-)": td >= "2020-06-01"}.items():
        for m in ["Persistence", "XGBoost"]:
            if msk.sum() > 3: add("periods", dict(city=city, period=pn, model=m, n=int(msk.sum()), **reg(yte[msk], pt[m][msk])))
fx = pd.DataFrame(R["fixed"]); fx.pivot(index="model", columns="city", values="test_RMSE").round(1)

## 5. Feature-group ablation (pollutant data) and multi-horizon forecasts

In [ ]:
for city, P in PREP.items():
    grp = {c: group_of(c, P["pol"]) for c in P["cols"]}
    sets = {"A0: AQI history only": [c for c in P["cols"] if grp[c] == "AQI history"],
            "A1: + pollutants": [c for c in P["cols"] if grp[c] == "AQI history" or grp[c] in P["pol"]],
            "A2: + calendar & flags": P["cols"]}
    for name, cs in sets.items():
        r = [reg(P["yte"], fit_xgb(DEF_XGB, P["Xtv_i"][cs], P["ytv"], s).predict(P["Xte_i2"][cs])) for s in range(2 if QUICK else 3)]
        add("ablation", dict(city=city, feature_set=name, n_features=len(cs), RMSE=np.mean([x["RMSE"] for x in r]), RMSE_std=np.std([x["RMSE"] for x in r]),
                             MAE=np.mean([x["MAE"] for x in r]), R2=np.mean([x["R2"] for x in r])))
    for h in [1, 3, 7]:
        Xh = build(P["d"], P["pol"], h); ch = feat_cols(Xh); a, b, c_ = split_masks(Xh)
        Xh_tv, Xh_te = impute(Xh.loc[a | b, ch], Xh.loc[c_, ch]); y_tv, y_te = Xh.loc[a | b, "y"].values, Xh.loc[c_, "y"].values
        pr = np.mean([fit_xgb(DEF_XGB, Xh_tv, y_tv, s).predict(Xh_te) for s in range(2 if QUICK else 3)], axis=0)
        add("horizon", dict(city=city, h=h, model="XGBoost", n=len(y_te), **reg(y_te, pr)))
        add("horizon", dict(city=city, h=h, model="Persistence", n=len(y_te), **reg(y_te, Xh.loc[c_, "AQI_lag0"].values)))
pd.DataFrame(R["ablation"]).round(2)

## 6. Walk-forward validation (expanding window, test years 2018-2020) and extreme-day (AQI > 300) analysis

In [ ]:
WF = {}
for city, P in PREP.items():
    X, cols = P["X"], P["cols"]; rows = []
    for Y in [2018, 2019, 2020]:
        trm = X["tdate"] < pd.Timestamp(f"{Y}-01-01"); tem = (X["tdate"] >= pd.Timestamp(f"{Y}-01-01")) & (X["tdate"] <= pd.Timestamp(f"{Y}-12-31"))
        if tem.sum() < 10 or trm.sum() < 200: continue
        a_, b_ = impute(X.loc[trm, cols], X.loc[tem, cols]); ya, yb = X.loc[trm, "y"].values, X.loc[tem, "y"].values
        Pm = {"Persistence": X.loc[tem, "AQI_lag0"].values, "Ridge (AR)": ridge_fit_predict(a_, ya, [b_], 10)[0],
              "Random Forest": rf_model(0).fit(a_, ya).predict(b_),
              "XGBoost": np.mean([fit_xgb(DEF_XGB, a_, ya, s).predict(b_) for s in range(3)], axis=0),
              "XGBoost-W (extreme-weighted)": np.mean([fit_xgb(DEF_XGB, a_, ya, s, w=1 + 3*(ya > 250)).predict(b_) for s in range(3)], axis=0)}
        for m, p in Pm.items(): rows.append(pd.DataFrame(dict(city=city, year=Y, model=m, tdate=X.loc[tem, "tdate"].values, y=yb, pred=p)))
    wf = pd.concat(rows); WF[city] = wf
    for m, g in wf.groupby("model"):
        e = g.y > EXTREME
        add("wf", dict(city=city, model=m, n=len(g), **reg(g.y, g.pred), **ext(g.y, g.pred),
                       ext_MAE=float(np.mean(np.abs(g.y[e] - g.pred[e]))) if e.sum() else np.nan, ext_bias=float(np.mean((g.pred - g.y)[e])) if e.sum() else np.nan))
pd.DataFrame(R["wf"])[["city", "model", "n", "RMSE", "R2", "support", "precision", "recall", "F1", "ext_MAE", "ext_bias"]].round(2)

## 7. SHAP, grouped by data source and by season (model trained on 2015-2018, explaining 2019-2020)

In [ ]:
import shap
for city, P in PREP.items():
    grp = {c: group_of(c, P["pol"]) for c in P["cols"]}
    Xa, Xb = impute(P["Xtr"], pd.concat([P["Xva"], P["Xte"]]))
    m = fit_xgb(DEF_XGB, Xa, P["ytr"], 0); sv = pd.DataFrame(np.abs(shap.TreeExplainer(m).shap_values(Xb)), columns=P["cols"], index=Xb.index)
    gsum = sv.T.groupby(lambda c: grp[c]).sum().T; imp = gsum.mean(); imp = imp / imp.sum() * 100
    for g_, v in imp.items(): add("shap", dict(city=city, group=g_, share_pct=float(v)))
    mth = pd.to_datetime(P["X"].loc[Xb.index, "tdate"]).dt.month
    seas = mth.map(lambda x: "Winter (Dec-Feb)" if x in (12, 1, 2) else "Pre-monsoon (Mar-May)" if x in (3, 4, 5) else "Monsoon (Jun-Sep)" if x in range(6, 10) else "Post-monsoon (Oct-Nov)")
    for s_, idx in gsum.groupby(seas):
        im = idx.mean(); im = im / im.sum() * 100
        for g_, v in im.items(): add("shap_season", dict(city=city, season=s_, group=g_, share_pct=float(v), n=len(idx)))
pd.DataFrame(R["shap"]).pivot(index="city", columns="group", values="share_pct").round(1)

## 8. Classical time-series baseline: ARIMA (one-step-ahead, parameters fitted on 2015-2019 only)

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
for city, P in PREP.items():
    s = P["d"]["AQI"].ffill(limit=3); s = s.fillna(s[:"2018-12-31"].median()); fit_s = s[:"2019-12-31"]
    try:
        res = SARIMAX(fit_s, order=(2, 0, 1), trend="c", enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=200)
        full = res.apply(s, refit=False)                                   # same parameters, filtered on the full series (no look-ahead)
        pred = full.get_prediction(start=pd.Timestamp("2020-01-01"), dynamic=False).predicted_mean      # one-step-ahead prediction for each 2020 day
        te_idx = P["X"].index[P["mte"]]; tgt = te_idx + pd.Timedelta(days=1)                                # forecast issued on day t for day t+1
        p = pred.reindex(tgt).values; y = P["X"].loc[P["mte"], "y"].values; ok = ~np.isnan(p)
        add("arima", dict(city=city, n=int(ok.sum()), **reg(y[ok], p[ok])))
    except Exception as e:
        add("arima", dict(city=city, n=0, RMSE=np.nan, MAE=np.nan, R2=np.nan))
pd.DataFrame(R["arima"]).round(2)

## 9. Fusion extension: ERA5 weather (feature set B), optional AOD / fire (sets C, D)
**9a** downloads daily ERA5 weather for the five cities from the Open-Meteo historical archive (ERA5 reanalysis, free, no API key; data licence CC BY 4.0 - cite Open-Meteo and ERA5 in the paper). It runs once and saves `era5_daily.csv`.
**9b** compares feature sets A (pollutants), B (+ weather) and, if you provide the files, C (+ AOD) and D (+ fire). **9c** repeats the comparison at 3- and 7-day horizons and adds the *perfect-forecast bound* B* (observed weather of the target day - an optimistic upper bound, not a deployable model).
All weather/AOD/fire variables enter as **day-t observations** (lag 0, lag 1, 3-day mean), never as next-day observations.

In [ ]:
# ---- 9a. Download daily ERA5 weather via Open-Meteo (skipped if era5_daily.csv already exists) ----
import requests, time
if os.path.exists(WEATHER_PATH):
    print("Found", WEATHER_PATH, "- skipping download")
else:
    CITY_XY = {"Delhi": (28.61, 77.21), "Lucknow": (26.85, 80.95), "Patna": (25.59, 85.14), "Chennai": (13.08, 80.27), "Hyderabad": (17.39, 78.49)}
    HOURLY = "temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure,precipitation"
    rows = []
    for city, (lat, lon) in CITY_XY.items():
        for yr in range(2015, 2021):
            end = "2020-07-01" if yr == 2020 else f"{yr}-12-31"
            r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=dict(latitude=lat, longitude=lon, start_date=f"{yr}-01-01", end_date=end,
                             hourly=HOURLY, timezone="Asia/Kolkata"), timeout=120)
            r.raise_for_status()
            h_ = pd.DataFrame(r.json()["hourly"]); h_["time"] = pd.to_datetime(h_["time"]); h_["City"] = city; rows.append(h_); time.sleep(2)
    h_ = pd.concat(rows); rad = np.deg2rad(h_["wind_direction_10m"])
    h_["u10"] = -h_["wind_speed_10m"] * np.sin(rad); h_["v10"] = -h_["wind_speed_10m"] * np.cos(rad); h_["Date"] = h_["time"].dt.floor("D")
    wx_daily = h_.groupby(["City", "Date"]).agg(t2m=("temperature_2m", "mean"), rh=("relative_humidity_2m", "mean"), ws=("wind_speed_10m", "mean"),
                u10=("u10", "mean"), v10=("v10", "mean"), sp=("surface_pressure", "mean"), tp=("precipitation", "sum")).reset_index()
    wx_daily.to_csv(WEATHER_PATH, index=False); print("Saved", WEATHER_PATH, wx_daily.shape)

In [ ]:
EXTRA_PREFIX = [("ERA5 weather", "wx_"), ("AOD", "aod_"), ("Fire", "fire_")]
def load_extra(path, prefix, city):
    e = pd.read_csv(path, parse_dates=["Date"]); e = e[e.City == city].drop(columns=["City"]).drop_duplicates("Date").set_index("Date").sort_index()
    e = e.reindex(pd.date_range(e.index.min(), e.index.max(), freq="D")); return e.add_prefix(prefix)
def make_extra(city, wx=True, aod=False, fire=False):
    parts = []
    if wx and os.path.exists(WEATHER_PATH): parts.append(load_extra(WEATHER_PATH, "wx_", city))
    if aod and os.path.exists(AOD_PATH): parts.append(load_extra(AOD_PATH, "aod_", city))
    if fire and os.path.exists(FIRE_PATH): parts.append(load_extra(FIRE_PATH, "fire_", city))
    if not parts: return None
    E = pd.concat(parts, axis=1).ffill(limit=3); F = {}
    for c in E.columns:
        F[f"{c}_lag0"], F[f"{c}_lag1"], F[f"{c}_r3"] = E[c], E[c].shift(1), E[c].rolling(3, min_periods=1).mean()
    return pd.DataFrame(F)

FUSION_SETS = {"A: pollutants only": dict(wx=False), "B: + ERA5 weather": dict(wx=True), "C: + AOD": dict(wx=True, aod=True), "D: + fire": dict(wx=True, aod=True, fire=True)}
FUS_PRED = {}
for city in CITY_LIST:
    for name, kw in FUSION_SETS.items():
        need = [p for flag, p in [("aod", AOD_PATH), ("fire", FIRE_PATH)] if kw.get(flag)]
        if any(not os.path.exists(p) for p in need): continue                  # skip sets whose input files are missing
        extra = make_extra(city, **kw) if any(kw.values()) else None
        if any(kw.values()) and extra is None: continue                      # file missing -> skip this set
        P = prepare(city, 1, extra); r, preds = [], []
        for s in range(NSEED):
            p = fit_xgb(DEF_XGB, P["Xtv_i"], P["ytv"], s).predict(P["Xte_i2"]); preds.append(p); r.append(reg(P["yte"], p))
        FUS_PRED[(city, name)] = (np.mean(preds, axis=0), P["yte"])
        add("fusion", dict(city=city, feature_set=name, n_features=len(P["cols"]), **reg(P["yte"], np.mean(preds, axis=0)), RMSE_std=np.std([x["RMSE"] for x in r]),
                           **{f"ext_{k}": v for k, v in ext(P["yte"], np.mean(preds, axis=0)).items()}))   # metrics of the seed-averaged forecast
        # walk-forward (pooled 2018-2020) for extreme-day metrics of each feature set
        X, cols = P["X"], P["cols"]; ys, ps = [], []
        for Y in [2018, 2019, 2020]:
            trm = X["tdate"] < pd.Timestamp(f"{Y}-01-01"); tem = (X["tdate"] >= pd.Timestamp(f"{Y}-01-01")) & (X["tdate"] <= pd.Timestamp(f"{Y}-12-31"))
            if tem.sum() < 10 or trm.sum() < 200: continue
            a_, b_ = impute(X.loc[trm, cols], X.loc[tem, cols]); ya, yb = X.loc[trm, "y"].values, X.loc[tem, "y"].values
            ps.append(np.mean([fit_xgb(DEF_XGB, a_, ya, s).predict(b_) for s in range(3)], axis=0)); ys.append(yb)
        ys, ps = np.concatenate(ys), np.concatenate(ps)
        add("fusion_wf", dict(city=city, feature_set=name, **reg(ys, ps), **ext(ys, ps)))
if "fusion" in R:
    fu = pd.DataFrame(R["fusion"]); display(fu.pivot(index="feature_set", columns="city", values="RMSE").round(2))
    for city in CITY_LIST:                                                   # Diebold-Mariano: does each fusion set beat pollutant-only?
        base = FUS_PRED.get((city, "A: pollutants only"))
        for name in FUSION_SETS:
            if name.startswith("A") or (city, name) not in FUS_PRED: continue
            p, y = FUS_PRED[(city, name)]; st, pv_ = dm_test(y - base[0], y - p, 1); add("fusion_dm", dict(city=city, feature_set=name, stat=st, p=pv_))
else:
    print("No weather/AOD/fire files found - section skipped.")

In [ ]:
# Source-wise SHAP for the richest fusion set available (grouped by pollutant / weather / AOD / fire / calendar)
if "fusion" in R:
    best = [k for k in FUSION_SETS if any(r["feature_set"] == k for r in R["fusion"])][-1]; kw = FUSION_SETS[best]
    for city in CITY_LIST:
        P = prepare(city, 1, make_extra(city, **kw)); grp = {c: group_of(c, P["pol"], EXTRA_PREFIX) for c in P["cols"]}
        src = {c: ("Pollutants & AQI history" if grp[c] in P["pol"] + ["AQI history"] else grp[c]) for c in P["cols"]}
        Xa, Xb = impute(P["Xtr"], pd.concat([P["Xva"], P["Xte"]])); m = fit_xgb(DEF_XGB, Xa, P["ytr"], 0)
        sv = pd.DataFrame(np.abs(shap.TreeExplainer(m).shap_values(Xb)), columns=P["cols"], index=Xb.index)
        g = sv.T.groupby(lambda c: src[c]).sum().T; imp = g.mean(); imp = imp / imp.sum() * 100
        for k_, v in imp.items(): add("fusion_shap", dict(city=city, feature_set=best, source=k_, share_pct=float(v)))
    display(pd.DataFrame(R["fusion_shap"]).pivot(index="city", columns="source", values="share_pct").round(1))

In [ ]:
# ---- 9c. Horizons 1/3/7 days: A (pollutants), B (day-t weather), B* (perfect-forecast bound: observed weather of the target day) ----
def make_extra_oracle(city, h):
    e = load_extra(WEATHER_PATH, "wxf_", city); F = {}
    for c in e.columns: F[f"{c}_h0"] = e[c].shift(-h); F[f"{c}_h1"] = e[c].shift(-h + 1) if h > 1 else e[c]
    return pd.DataFrame(F)
if os.path.exists(WEATHER_PATH):
    for city in CITY_LIST:
        d_, pol_ = PREP[city]["d"], PREP[city]["pol"]
        for h in [1, 3, 7]:
            for name, ex in [("A: pollutants only", None), ("B: + ERA5 weather (day t)", make_extra(city, wx=True)), ("B*: perfect weather forecast (upper bound)", make_extra_oracle(city, h))]:
                Xh = build(d_, pol_, h, extra=ex); ch = feat_cols(Xh); a, b, c_ = split_masks(Xh)
                tv, te = impute(Xh.loc[a | b, ch], Xh.loc[c_, ch]); ytv_, yte_ = Xh.loc[a | b, "y"].values, Xh.loc[c_, "y"].values
                p = np.mean([fit_xgb(DEF_XGB, tv, ytv_, s).predict(te) for s in range(2 if QUICK else 3)], axis=0); pr = ridge_fit_predict(tv, ytv_, [te], 10)[0]
                add("fusion_horizon", dict(city=city, h=h, feature_set=name, model="XGBoost", n=len(yte_), **reg(yte_, p)))
                add("fusion_horizon", dict(city=city, h=h, feature_set=name, model="Ridge (AR)", n=len(yte_), **reg(yte_, pr)))
            add("fusion_horizon", dict(city=city, h=h, feature_set="Persistence", model="Persistence", n=len(yte_), **reg(yte_, Xh.loc[c_, "AQI_lag0"].values)))
    display(pd.DataFrame(R["fusion_horizon"]).pivot_table(index=["model", "feature_set", "h"], columns="city", values="RMSE").round(1))

## 10. Optional: GRU sequence model (PyTorch; use a GPU runtime for speed)

In [ ]:
try:
    import torch, torch.nn as nn
    HAS_TORCH = True
except Exception:
    HAS_TORCH = False; print("PyTorch not available - skipping GRU")

def gru_city(P, L=14, seeds=3, epochs=120, hidden=64):
    d, pol = P["d"], P["pol"]; f = d[["AQI"] + pol].ffill(limit=3); f = f.fillna(f[:T_TRAIN_END].median())
    mu, sd_ = f[:T_TRAIN_END].mean(), f[:T_TRAIN_END].std() + 1e-6; Z = ((f - mu) / sd_).values.astype(np.float32); pos = {t: i for i, t in enumerate(f.index)}
    def seqs(Xdf):
        out = []
        for t in Xdf.index:
            i = pos[t]; w = Z[max(0, i - L + 1): i + 1]
            if len(w) < L: w = np.vstack([np.repeat(w[:1], L - len(w), 0), w])
            out.append(w)
        return torch.tensor(np.array(out, dtype=np.float32))
    Str, Sva, Ste = seqs(P["Xtr"]), seqs(P["Xva"]), seqs(P["Xte"]); ym, ys = P["ytr"].mean(), P["ytr"].std()
    ytr = torch.tensor(((P["ytr"] - ym) / ys).astype(np.float32)); res = []
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    for s in range(seeds):
        torch.manual_seed(s)
        class Net(nn.Module):
            def __init__(self): super().__init__(); self.g = nn.GRU(Str.shape[2], hidden, batch_first=True); self.o = nn.Linear(hidden, 1)
            def forward(self, x): return self.o(self.g(x)[0][:, -1]).squeeze(-1)
        net = Net().to(dev); opt = torch.optim.Adam(net.parameters(), 1e-3); best, best_state, bad = 1e18, None, 0
        for ep in range(epochs):
            net.train(); perm = torch.randperm(len(Str))
            for i in range(0, len(Str), 64):
                idx = perm[i:i + 64]; opt.zero_grad(); loss = ((net(Str[idx].to(dev)) - ytr[idx].to(dev)) ** 2).mean(); loss.backward(); opt.step()
            net.eval()
            with torch.no_grad(): v = float(np.sqrt(mean_squared_error(P["yva"], net(Sva.to(dev)).cpu().numpy() * ys + ym)))
            if v < best: best, bad, best_state = v, 0, {k: x.clone() for k, x in net.state_dict().items()}
            else:
                bad += 1
                if bad >= 15: break
        net.load_state_dict(best_state); net.eval()
        with torch.no_grad(): res.append(net(Ste.to(dev)).cpu().numpy() * ys + ym)
    return np.mean(res, axis=0), [reg(P["yte"], r) for r in res]

if HAS_TORCH:
    for city, P in PREP.items():
        p, rr = gru_city(P, seeds=2 if QUICK else 3); add("gru", dict(city=city, **reg(P["yte"], p), RMSE_std=float(np.std([x["RMSE"] for x in rr]))))
    display(pd.DataFrame(R["gru"]).round(2))

## 11. Save all result tables

In [ ]:
for k, v in R.items(): pd.DataFrame(v).to_csv(f"{OUT}/{k}.csv", index=False)
print("Saved:", sorted(R.keys()), "->", OUT)